In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch
from scipy.interpolate import make_interp_spline
import warnings
warnings.filterwarnings('ignore')
#path to repo: 

repository= "https://github.com/GoogleCloudPlatform/covid-19-open-data/blob/main/docs/table-search-trends.md"
# Load Google symptom search trends
google_searches = pd.read_csv('google-search-trends (1).csv',low_memory=False) #replace this with your relative path to the file
google_searches['date'] = pd.to_datetime(google_searches['date'])
print(f"Shape: {google_searches.shape}") #explore shape of the data
print(f"Date range: {google_searches['date'].min()} → {google_searches['date'].max()}") #Explore date range of the data

# Extract symptom columns
symptom_cols = [c for c in google_searches.columns if c.startswith('search_trends_')]
symptom_names = [c.replace('search_trends_', '').replace('_', ' ').title() for c in symptom_cols]
print(f"Number of symptoms: {len(symptom_cols)}")

Shape: (2713929, 424)
Date range: 2020-01-01 00:00:00 → 2022-09-12 00:00:00
Number of symptoms: 422


In [9]:
set(symptom_cols)

{'search_trends_abdominal_obesity',
 'search_trends_abdominal_pain',
 'search_trends_acne',
 'search_trends_actinic_keratosis',
 'search_trends_acute_bronchitis',
 'search_trends_adrenal_crisis',
 'search_trends_ageusia',
 'search_trends_alcoholism',
 'search_trends_allergic_conjunctivitis',
 'search_trends_allergy',
 'search_trends_amblyopia',
 'search_trends_amenorrhea',
 'search_trends_amnesia',
 'search_trends_anal_fissure',
 'search_trends_anaphylaxis',
 'search_trends_anemia',
 'search_trends_angina_pectoris',
 'search_trends_angioedema',
 'search_trends_angular_cheilitis',
 'search_trends_anosmia',
 'search_trends_anxiety',
 'search_trends_aphasia',
 'search_trends_aphonia',
 'search_trends_apnea',
 'search_trends_arthralgia',
 'search_trends_arthritis',
 'search_trends_ascites',
 'search_trends_asperger_syndrome',
 'search_trends_asphyxia',
 'search_trends_asthma',
 'search_trends_astigmatism',
 'search_trends_ataxia',
 'search_trends_atheroma',
 'search_trends_attention_defici

In [10]:
# --- let's categorize symptoms into 10 medical system groups ---
# Expanded keyword lists to capture more of the 422 symptoms
# Note: assign_category checks if ANY keyword is a substring of the symptom name
# Order matters — first match wins
categories_v2 = {
    'Respiratory': ['cough', 'breath', 'pneumonia', 'nasal', 'sinus', 'wheez', 'bronch',
                    'asthma', 'sneez', 'phlegm', 'lung', 'airway', 'throat', 'laryn',
                    'stridor', 'sputum', 'apnea', 'tachypnea', 'pleur', 'croup',
                    'rhinit', 'rhinor', 'tonsil', 'hyperventil', 'crackle', 'hypoxia',
                    'hypoxem', 'hypercapnia', 'pulmonary', 'snor'],
    'Neurological': ['headache', 'dizz', 'vertigo', 'seizure', 'migraine', 'neuropath',
                     'brain', 'nerve', 'numbness', 'paraly', 'tremor', 'confusion', 'memory',
                     'ataxia', 'epilep', 'convuls', 'encephalit', 'encephalop', 'meningit',
                     'nystagm', 'aphasi', 'aphoni', 'dystoni', 'myoclon', 'chorea',
                     'cataplexy', 'radiculop', 'sciatica', 'intracranial', 'dementia',
                     'subdural', 'fascicul', 'hydrocephalus', 'unconscious', 'coma',
                     'lightheaded', 'paresthesia', 'syncope', 'polyneuropath', 'ptosis'],
    'Gastrointestinal': ['nausea', 'vomit', 'diarr', 'stomach', 'abdomin', 'bowel', 'digest',
                         'constip', 'bloat', 'gastro', 'intestin', 'appetite', 'reflux',
                         'colitis', 'esophag', 'pancreatit', 'flatulen', 'fecal', 'rectal',
                         'melena', 'hematochezi', 'biliary', 'malabsorp', 'dysphagia',
                         'hiccup', 'hemorrhoid', 'blood_in_stool', 'morning_sickness',
                         'motion_sickness', 'lactose', 'celiac', 'globus_pharyn'],
    'Musculoskeletal': ['pain', 'muscle', 'joint', 'back_pain', 'bone', 'arthri', 'spasm',
                        'cramp', 'stiff', 'weak', 'body_ache', 'sore', 'fibro', 'inflam',
                        'scoli', 'kypho', 'osteop', 'bunion', 'tendern', 'gout',
                        'crepitus', 'posture', 'hypermobil', 'carpal', 'restless_leg',
                        'myalg', 'spastic', 'fracture', 'podalgia'],
    'Cardiovascular': ['heart', 'chest', 'blood_pressure', 'pulse', 'cardio', 'palpitat',
                       'hypertens', 'arrhyth', 'stroke', 'circulat', 'vascular', 'tachycard',
                       'bradycard', 'fibrillat', 'angina', 'atheroma', 'myocardial',
                       'cardiac', 'pericarditis', 'varicose', 'cyanosis', 'hypotens',
                       'mitral', 'congenital_heart', 'ventricular'],
    'Dermatological': ['rash', 'skin', 'itch', 'hive', 'blister', 'lesion', 'dermat',
                       'eczema', 'acne', 'burn', 'swelling', 'wound', 'bruise',
                       'wart', 'dandruff', 'hair_loss', 'scar', 'stretch_mark', 'papule',
                       'nodule', 'cyst', 'erythem', 'rosacea', 'xeroderma', 'impetigo',
                       'petechia', 'purpura', 'telangiect', 'hyperpigment', 'desquamat',
                       'cheilitis', 'ingrown', 'boil', 'candidiasis', 'pus', 'blush',
                       'photodermat', 'milium', 'onychorrhexis', 'trichoptilosis',
                       'beaus_lines', 'melasma', 'granuloma', 'chancre'],
    'Psychological': ['anxiety', 'depress', 'insomnia', 'sleep', 'stress', 'mood',
                      'panic', 'fatigue', 'tired', 'mental', 'lethar', 'irritab',
                      'compulsiv', 'paranoia', 'psychosis', 'suicid', 'self_harm',
                      'binge', 'hallucinat', 'depersonaliz', 'dysphori', 'hypochondri',
                      'grandiosity', 'impulsiv', 'rumin', 'manic', 'avoidant',
                      'attention_deficit', 'guilt', 'shyness', 'bruxism', 'hypomania',
                      'night_terror', 'stuttering'],
    'ENT & Sensory': ['taste', 'smell', 'ear', 'hear', 'tinnit', 'eye', 'vision',
                      'nose', 'voice', 'hoarse', 'deaf', 'conjunct', 'anosmia',
                      'otitis', 'photophob', 'photopsia', 'floater', 'strabismus',
                      'epiphora', 'cataract', 'amblyopia', 'astigmat', 'visual',
                      'geusia', 'blepharospasm', 'xerostomia', 'halitosis',
                      'sensitivity_to_sound', 'nosebleed', 'red_eye', 'dry_eye'],
    'Immune & Systemic': ['fever', 'chills', 'sweat', 'lymph', 'immune', 'influen',
                          'infect', 'malaise', 'sepsis', 'autoimmun', 'allerg',
                          'anaphylax', 'neutropenia', 'thrombocytopenia', 'anemia',
                          'polycythemia', 'hemolysis', 'splenomeg', 'shiver',
                          'iron_deficiency', 'folate', 'hot_flash', 'hemoptysis',
                          'bleeding'],
    'Metabolic & Other': ['weight', 'diabetes', 'thirst', 'urin', 'kidney', 'liver',
                          'thyroid', 'dehydrat', 'edema', 'sugar', 'metabol',
                          'obesity', 'hyperglycemia', 'hypoglycemia', 'insulin',
                          'prediabet', 'ketoacid', 'hypercholesterol', 'hyperlipid',
                          'hypertriglyc', 'goitre', 'uria', 'hypercalcaemia',
                          'hyperkalemia', 'hypokalemia', 'hyponatremia', 'hypocalcaemia',
                          'polydipsia', 'jaundice', 'cirrhosis', 'hepat',
                          'ascites', 'proteinuria', 'pyelonephritis', 'renal']
}

def assign_category(col_name):
    name = col_name.replace('search_trends_', '').lower()
    for cat, keywords in categories_v2.items():
        if any(kw in name for kw in keywords):
            return cat
    return None  # uncategorized

symptom_category_map = {col: assign_category(col) for col in symptom_cols}
categorized = {col: cat for col, cat in symptom_category_map.items() if cat is not None}
cat_list = list(categories_v2.keys())

print(f"Categorized: {len(categorized)}/{len(symptom_cols)} symptoms")
for cat in cat_list:
    count = sum(1 for v in categorized.values() if v == cat)
    print(f"  {cat}: {count}")

Categorized: 365/422 symptoms
  Respiratory: 36
  Neurological: 40
  Gastrointestinal: 32
  Musculoskeletal: 47
  Cardiovascular: 23
  Dermatological: 53
  Psychological: 36
  ENT & Sensory: 26
  Immune & Systemic: 29
  Metabolic & Other: 43


In [11]:
# Convert symptom columns to numeric
for col in categorized.keys():
    google_searches[col] = pd.to_numeric(google_searches[col], errors='coerce')

# Monthly aggregation
monthly = google_searches.set_index('date')[list(categorized.keys())].resample('M').mean()

# Build category-level monthly data
cat_monthly = pd.DataFrame(index=monthly.index)
for cat in cat_list:
    cols_in_cat = [c for c, v in categorized.items() if v == cat]
    cat_monthly[cat] = monthly[cols_in_cat].sum(axis=1)

cat_monthly = cat_monthly.fillna(0)
symptom_monthly = monthly.fillna(0)

print(f"Monthly time points: {len(cat_monthly)}")
print(f"Date range: {cat_monthly.index[0]} → {cat_monthly.index[-1]}")
print(f"\nCategory totals (mean monthly volume):")
print(cat_monthly.mean().sort_values(ascending=False))

Monthly time points: 33
Date range: 2020-01-31 00:00:00 → 2022-09-30 00:00:00

Category totals (mean monthly volume):
Musculoskeletal      80.654319
Dermatological       67.106936
Immune & Systemic    61.297081
Metabolic & Other    41.531231
Psychological        41.185411
Gastrointestinal     37.253851
Neurological         27.616807
Respiratory          24.106900
Cardiovascular       23.849100
ENT & Sensory        10.208498
dtype: float64


In [12]:
# --- Compute streamgraph layout with wiggle offset ---
n_cats = len(cat_list)
n_time = len(cat_monthly)

data = cat_monthly[cat_list].values.T  # (10, 33)

def wiggle_offset(data):
    """Compute stacked layout with wiggle/silhouette offset."""
    n, m = data.shape  
    cumsum = np.cumsum(data, axis=0)
    total = cumsum[-1]
    
    offset = -total / 2
    
    lower = np.zeros_like(data)
    upper = np.zeros_like(data)
    
    lower[0] = offset
    upper[0] = offset + data[0]
    
    for i in range(1, n):
        lower[i] = upper[i-1]
        upper[i] = lower[i] + data[i]
    
    return lower, upper

lower, upper = wiggle_offset(data)

from scipy.ndimage import uniform_filter1d

smooth_lower = np.array([uniform_filter1d(lower[i], size=3) for i in range(n_cats)])
smooth_upper = np.array([uniform_filter1d(upper[i], size=3) for i in range(n_cats)])

x_positions = np.arange(n_time)
x_dates = cat_monthly.index

# Color palette for categories
color_palette = {
    'Respiratory': '#E63946',
    'Neurological': '#457B9D',
    'Gastrointestinal': '#2A9D8F',
    'Musculoskeletal': '#E9C46A',
    'Cardiovascular': '#F4A261',
    'Dermatological': '#264653',
    'Psychological': '#8338EC',
    'ENT & Sensory': '#06D6A0',
    'Immune & Systemic': '#EF476F',
    'Metabolic & Other': '#118AB2'
}

In [13]:
# =====================================================
# TRUE WORDSTREAM: Word-packing inside stream bands
# =====================================================
# 
# Key idea: Each stream band is divided into time-step boxes.
# Words are placed inside each box using spiral placement
# with bitmap collision detection - just like a word cloud
# but constrained to the stream band shape.

from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure as MplFigure
from PIL import Image, ImageDraw, ImageFont
import io

# --- Step 1: Get top symptoms per category per time step ---
def get_top_symptoms_per_cat_time(symptom_monthly, categorized, cat_list, top_n=8):
    """For each category and each time step, get the top-N symptoms by volume."""
    result = {}
    for cat in cat_list:
        cols_in_cat = [c for c, v in categorized.items() if v == cat]
        cat_data = symptom_monthly[cols_in_cat]
        result[cat] = []
        for t_idx in range(len(cat_data)):
            row = cat_data.iloc[t_idx]
            top = row.nlargest(top_n)
            # Clean names
            words = []
            for col, val in top.items():
                if val > 0:
                    name = col.replace('search_trends_', '').replace('_', ' ').title()
                    # Shorten long names
                    if len(name) > 12:
                        name = name[:11] + '.'
                    words.append((name, val))
            result[cat].append(words)
    return result

top_words = get_top_symptoms_per_cat_time(symptom_monthly, categorized, cat_list, top_n=10)

Sample: Respiratory, month 0 (Jan 2020):
  Cough: 8.61
  Sore Throat: 2.48
  Nasal Conge.: 2.33
  Sinusitis: 2.22
  Asthma: 2.17
  Pneumonia: 1.86
  Sleep Apnea: 1.34
  Snoring: 1.07
  Bronchitis: 1.02
  Phlegm: 0.85


In [14]:
# --- WordStream Word-Packing Class as engine ---

class WordStreamPacker:
    """
    Places words inside stream band regions using spiral placement
    and pixel-level collision detection, mimicking the original
    WordStream algorithm (buildBoard + place + cloudCollide).
    """
    
    def __init__(self, fig_width_px, fig_height_px, dpi=100):
        self.width = fig_width_px
        self.height = fig_height_px
        self.dpi = dpi
        # Global bitmap: 0 = free, 1 = occupied
        self.bitmap = np.zeros((fig_height_px, fig_width_px), dtype=np.uint8)
        self.placed_words = []
    
    def _measure_text(self, text, fontsize):
        """Measure text bounding box in pixels."""
        char_width = fontsize * 0.58
        w = int(len(text) * char_width) + 4
        h = int(fontsize * 1.2) + 2
        return w, h
    
    def _archimedean_spiral(self, center_x, center_y, max_radius):
        """Generate positions along an Archimedean spiral."""
        t = 0
        dt = 1
        a = 0.5  
        while True:
            r = a * t
            if r > max_radius:
                break
            x = int(center_x + r * np.cos(t))
            y = int(center_y + r * np.sin(t))
            yield x, y
            t += dt
    
    def _check_collision(self, x, y, w, h):
        """Check if placing text at (x,y) with size (w,h) collides."""
        x0 = x - w // 2
        y0 = y - h // 2
        x1 = x0 + w
        y1 = y0 + h
        
        # Boundary check
        if x0 < 0 or y0 < 0 or x1 >= self.width or y1 >= self.height:
            return True
        
        # Check bitmap region
        region = self.bitmap[y0:y1, x0:x1]
        return np.any(region != 0)
    
    def _place_on_bitmap(self, x, y, w, h):
        """Mark region as occupied."""
        x0 = x - w // 2
        y0 = y - h // 2
        self.bitmap[y0:y0+h, x0:x0+w] = 1
    
    def fill_band_mask(self, lower_curve, upper_curve, x_pixel_positions):
        """
        Mark pixels OUTSIDE the stream band as occupied (=1).
        This constrains word placement to INSIDE the band shape.
        """
        for col_idx, x_px in enumerate(x_pixel_positions):
            if col_idx >= len(lower_curve) or col_idx >= len(upper_curve):
                continue
            y_low = int(lower_curve[col_idx])
            y_up = int(upper_curve[col_idx])
            
            if col_idx < len(x_pixel_positions) - 1:
                next_x = x_pixel_positions[col_idx + 1]
                for px in range(int(x_px), int(next_x)):
                    if 0 <= px < self.width:
                        frac = (px - x_px) / max(1, next_x - x_px)
                        if col_idx < len(lower_curve) - 1:
                            y_l = int(lower_curve[col_idx] * (1-frac) + lower_curve[col_idx+1] * frac)
                            y_u = int(upper_curve[col_idx] * (1-frac) + upper_curve[col_idx+1] * frac)
                        else:
                            y_l, y_u = y_low, y_up
                        
                        y_l = max(0, min(self.height-1, y_l))
                        y_u = max(0, min(self.height-1, y_u))
                        
                        self.bitmap[0:y_l, px] = 1
                        self.bitmap[y_u:, px] = 1
    
    def place_word(self, text, fontsize, center_x, center_y, band_height, color, category):
        """Try to place a word near (center_x, center_y) using spiral search."""
        w, h = self._measure_text(text, fontsize)
        
        if h > band_height * 0.9:
            return False
        
        max_radius = max(band_height, 80)
        
        for sx, sy in self._archimedean_spiral(center_x, center_y, max_radius):
            if not self._check_collision(sx, sy, w, h):
                self._place_on_bitmap(sx, sy, w, h)
                self.placed_words.append({
                    'text': text,
                    'x': sx,
                    'y': sy,
                    'fontsize': fontsize,
                    'color': color,
                    'category': category,
                    'w': w,
                    'h': h
                })
                return True
        return False

print("WordStreamPacker engine ready.")

WordStreamPacker engine ready.


In [15]:
# =====================================================
# RENDER THE WORDSTREAM - Fixed per-band placement
# =====================================================

FIG_W, FIG_H = 22, 10
DPI = 100
PX_W, PX_H = int(FIG_W * DPI), int(FIG_H * DPI)

x_margin = 60
y_margin = 40

y_min_data = smooth_lower.min()
y_max_data = smooth_upper.max()
y_range_data = y_max_data - y_min_data

def data_to_px_x(t_idx):
    return x_margin + t_idx * (PX_W - 2*x_margin) / (n_time - 1)

def data_to_px_y(y_val):
    normalized = (y_val - y_min_data) / y_range_data
    return int(PX_H - y_margin - normalized * (PX_H - 2*y_margin))

all_volumes = []
for cat in cat_list:
    for t_words in top_words[cat]:
        for _, vol in t_words:
            if vol > 0:
                all_volumes.append(vol)

vol_min, vol_max = np.percentile(all_volumes, 10), np.percentile(all_volumes, 95)
FONT_MIN, FONT_MAX = 6, 24

def volume_to_fontsize(vol):
    normalized = np.clip((vol - vol_min) / (vol_max - vol_min), 0, 1)
    return FONT_MIN + normalized * (FONT_MAX - FONT_MIN)

all_placed_words = []

for i, cat in enumerate(cat_list):
    color = color_palette[cat]
    
        lower_px_arr = np.array([data_to_px_y(smooth_upper[i, t]) for t in range(n_time)])
    upper_px_arr = np.array([data_to_px_y(smooth_lower[i, t]) for t in range(n_time)])
    x_px_arr = np.array([data_to_px_x(t) for t in range(n_time)])
    
    band_bitmap = np.ones((PX_H, PX_W), dtype=np.uint8) 
    
    for t_idx in range(n_time - 1):
        x_start = int(x_px_arr[t_idx])
        x_end = int(x_px_arr[t_idx + 1])
        for px in range(max(0, x_start), min(PX_W, x_end)):
            frac = (px - x_start) / max(1, x_end - x_start)
            y_top = int(lower_px_arr[t_idx] * (1-frac) + lower_px_arr[t_idx+1] * frac)
            y_bot = int(upper_px_arr[t_idx] * (1-frac) + upper_px_arr[t_idx+1] * frac)
            y_top = max(0, min(PX_H-1, y_top))
            y_bot = max(0, min(PX_H-1, y_bot))
            if y_top < y_bot:
                band_bitmap[y_top:y_bot, px] = 0
    
    t_last = n_time - 1
    x_last = int(x_px_arr[t_last])
    y_top_last = int(lower_px_arr[t_last])
    y_bot_last = int(upper_px_arr[t_last])
    for px in range(max(0, x_last), min(PX_W, x_last + 20)):
        if y_top_last < y_bot_last:
            band_bitmap[y_top_last:y_bot_last, px] = 0
    
    free_px = (band_bitmap == 0).sum()
    

    recent_words = {}  
    DEDUP_WINDOW = 3   # minimum months gap before same word can reappear
    
    band_placed = 0
    for t_idx in range(n_time):
        words_at_t = top_words[cat][t_idx]
        if not words_at_t:
            continue
        
        center_x = int(x_px_arr[t_idx])
        y_top = int(lower_px_arr[t_idx])
        y_bot = int(upper_px_arr[t_idx])
        center_y = (y_top + y_bot) // 2
        band_h = y_bot - y_top
        
        if band_h < 8:
            continue
        
        for word, vol in sorted(words_at_t, key=lambda x: -x[1]):
            if word in recent_words and (t_idx - recent_words[word]) < DEDUP_WINDOW:
                continue
            fontsize = volume_to_fontsize(vol)
            char_w = fontsize * 0.58
            w = int(len(word) * char_w) + 4
            h = int(fontsize * 1.2) + 2
            
            if h > band_h - 2:
                fontsize = max(FONT_MIN, (band_h - 4) / 1.2)
                w = int(len(word) * fontsize * 0.58) + 4
                h = int(fontsize * 1.2) + 2
            
            if h > band_h - 2 or fontsize < FONT_MIN:
                continue
            
            placed = False
            t = 0
            dt = 0.8
            max_r = max(band_h * 2, 100)
            
            while t < 200:
                r = 0.8 * t
                if r > max_r:
                    break
                sx = int(center_x + r * np.cos(t))
                sy = int(center_y + r * np.sin(t))
                t += dt
                
                x0 = sx - w // 2
                y0 = sy - h // 2
                x1 = x0 + w
                y1 = y0 + h
                
                if x0 < 0 or y0 < 0 or x1 >= PX_W or y1 >= PX_H:
                    continue
                
                if not np.any(band_bitmap[y0:y1, x0:x1]):
                    band_bitmap[y0:y1, x0:x1] = 1
                    all_placed_words.append({
                        'text': word, 'x': sx, 'y': sy,
                        'fontsize': fontsize, 'color': color,
                        'category': cat, 'vol': vol, 't_idx': t_idx
                    })
                    band_placed += 1
                    recent_words[word] = t_idx
                    placed = True
                    break
            
    print(f"  {cat}: placed {band_placed} words (band area: {free_px} free px)")

print(f"\nTotal placed: {len(all_placed_words)} words")

  Respiratory: placed 103 words (band area: 105851 free px)
  Neurological: placed 105 words (band area: 121482 free px)
  Gastrointestinal: placed 90 words (band area: 164036 free px)
  Musculoskeletal: placed 110 words (band area: 354919 free px)
  Cardiovascular: placed 86 words (band area: 104995 free px)
  Dermatological: placed 113 words (band area: 295203 free px)
  Psychological: placed 91 words (band area: 181496 free px)
  ENT & Sensory: placed 57 words (band area: 44876 free px)
  Immune & Systemic: placed 114 words (band area: 269695 free px)
  Metabolic & Other: placed 99 words (band area: 182882 free px)

Total placed: 968 words


In [ ]:
# =====================================================
# INTERACTIVE WORDSTREAM (Plotly) — with hover & legend toggle
# =====================================================
import plotly.graph_objects as go

def px_to_data_x(px_x):
    return (px_x - x_margin) * (n_time - 1) / (PX_W - 2 * x_margin)

def px_to_data_y(px_y):
    normalized = (PX_H - y_margin - px_y) / (PX_H - 2 * y_margin)
    return y_min_data + normalized * y_range_data

def time_idx_to_date(t_idx):
    t_int = int(np.clip(t_idx, 0, n_time - 1))
    t_frac = t_idx - t_int
    if t_int >= n_time - 1:
        return x_dates[-1]
    d1 = x_dates[t_int]
    d2 = x_dates[min(t_int + 1, n_time - 1)]
    return d1 + (d2 - d1) * t_frac

# Helper: hex color to rgba string
def hex_to_rgba(hex_color, alpha=1.0):
    r = int(hex_color[1:3], 16)
    g = int(hex_color[3:5], 16)
    b = int(hex_color[5:7], 16)
    return f"rgba({r},{g},{b},{alpha})"

fig = go.Figure()

# 1) Stream band fills
for i, cat in enumerate(cat_list):
    x_time = [x_dates[t] for t in range(n_time)]
    y_lo = [smooth_lower[i, t] for t in range(n_time)]
    y_up = [smooth_upper[i, t] for t in range(n_time)]
    
    fig.add_trace(go.Scatter(
        x=x_time, y=y_lo,
        mode='lines', line=dict(width=0),
        showlegend=False, hoverinfo='skip',
        legendgroup=cat
    ))
    fig.add_trace(go.Scatter(
        x=x_time, y=y_up,
        mode='lines', line=dict(width=0.3, color=hex_to_rgba(color_palette[cat], 0.4)),
        fill='tonexty',
        fillcolor=hex_to_rgba(color_palette[cat], 0.10),
        name=cat,
        legendgroup=cat,
        showlegend=True,
        hoverinfo='skip'
    ))

# 2) Words with quantitative hover tooltips
for cat in cat_list:
    cat_words = [w for w in all_placed_words if w['category'] == cat]
    if not cat_words:
        continue
    
    word_x, word_y, word_texts, word_sizes, word_hovers = [], [], [], [], []
    
    # Precompute category totals per month for % share
    cat_cols = [c for c, v in categorized.items() if v == cat]
    cat_totals = symptom_monthly[cat_cols].sum(axis=1).values
    
    for w in cat_words:
        t_idx_val = px_to_data_x(w['x'])
        y_val = px_to_data_y(w['y'])
        date = time_idx_to_date(t_idx_val)
        
        word_x.append(date)
        word_y.append(y_val)
        word_texts.append(w['text'])
        word_sizes.append(max(7, w['fontsize'] * 0.9))
        
        # Quantitative info
        vol = w.get('vol', 0)
        t_int = int(np.clip(w.get('t_idx', 0), 0, n_time - 1))
        cat_total = cat_totals[t_int] if t_int < len(cat_totals) else 1
        pct_share = (vol / cat_total * 100) if cat_total > 0 else 0
        
        # Rank within category for this time step
        words_this_t = top_words[cat][t_int]
        sorted_words = sorted(words_this_t, key=lambda x: -x[1])
        rank = next((idx+1 for idx, (nm, _) in enumerate(sorted_words) if nm == w['text']), '—')
        
        word_hovers.append(
            f"<b>{w['text']}</b><br>"
            f"Category: {cat}<br>"
            f"Period: {date.strftime('%b %Y')}<br>"
            f"─────────────<br>"
            f"Search volume: <b>{vol:.2f}</b><br>"
            f"Share of category: <b>{pct_share:.1f}%</b><br>"
            f"Category total: {cat_total:.1f}<br>"
            f"Rank in category: <b>#{rank}</b>"
        )
    
    # Text trace (the visible words)
    fig.add_trace(go.Scatter(
        x=word_x, y=word_y,
        mode='text',
        text=word_texts,
        textfont=dict(
            size=word_sizes,
            color=hex_to_rgba(color_palette[cat], 0.85),
            family='Arial Black, Arial'
        ),
        hoverinfo='skip',
        legendgroup=cat,
        showlegend=False,
        name=cat
    ))
    
    # Invisible marker trace for hover interaction
    fig.add_trace(go.Scatter(
        x=word_x, y=word_y,
        mode='markers',
        marker=dict(size=word_sizes, color='rgba(0,0,0,0)', line=dict(width=0)),
        hovertext=word_hovers,
        hoverinfo='text',
        hoverlabel=dict(
            bgcolor=hex_to_rgba(color_palette[cat], 0.9),
            font_size=12,
            font_color='white'
        ),
        legendgroup=cat,
        showlegend=False,
        name=cat
    ))

# Layout
fig.update_layout(
    title=dict(
        text='<b>Interactive WordStream</b>: COVID-19 Symptom Search Trends',
        font=dict(size=15),
        x=0.5
    ),
    xaxis=dict(
        showgrid=True, gridcolor='#f5f5f5', gridwidth=0.5,
        tickformat='%b<br>%Y', dtick='M3',
        showline=False
    ),
    yaxis=dict(
        showgrid=False, zeroline=False,
        showticklabels=False, showline=False
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    hovermode='closest',
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.12,
        xanchor='center', x=0.5, font=dict(size=10),
        tracegroupgap=5
    ),
    width=1800, height=900,
    margin=dict(l=20, r=20, t=60, b=90),
    annotations=[dict(
        text="<i>Hover words for details · Click legend to toggle categories · Drag to zoom · Double-click to reset</i>",
        xref="paper", yref="paper", x=0.5, y=-0.18,
        showarrow=False, font=dict(size=9, color='#999')
    )]
)

fig.show()